In [3]:
import io
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pywt
import scipy.fftpack
from IPython.display import display, clear_output
import ipywidgets as widgets
from skimage.metrics import structural_similarity as ssim

# ==== Utility Functions ====
def text_to_bits(text):
    return ''.join(format(ord(c), '08b') for c in text) + '1111111111111110'

def bits_to_text(bits):
    chars = [bits[i:i+8] for i in range(0, len(bits), 8)]
    text = ''
    for c in chars:
        if c == '11111111' or c == '1111111111111110':
            break
        text += chr(int(c, 2))
    return text

# ==== LSB ====
def embed_lsb(image, watermark_text):
    bits = text_to_bits(watermark_text)
    data_index = 0
    h, w, _ = image.shape
    img_copy = image.copy()
    for row in range(h):
        for col in range(w):
            for ch in range(3):
                if data_index < len(bits):
                    img_copy[row, col, ch] = (img_copy[row, col, ch] & 254) | int(bits[data_index])
                    data_index += 1
    return img_copy

def extract_lsb(image):
    bits = ''
    h, w, _ = image.shape
    for row in range(h):
        for col in range(w):
            for ch in range(3):
                bits += str(image[row, col, ch] & 1)
                if bits.endswith('1111111111111110'):
                    return bits_to_text(bits)
    return bits_to_text(bits)

# Cài đặt thuật toán LBS (Least Bit Substitution) – có thể chọn vị trí bit để nhúng (1 = LSB, 2 = bit thứ 2, ...)
def embed_lbs(image, watermark_text, bit_position=1):
    bits = text_to_bits(watermark_text)
    data_index = 0
    img_copy = image.copy()
    h, w, _ = img_copy.shape

    if not (1 <= bit_position <= 8):
        raise ValueError("Vị trí bit phải nằm trong khoảng 1 đến 8.")

    mask = ~(1 << (bit_position - 1))
    for y in range(h):
        for x in range(w):
            for c in range(3):  # R, G, B
                if data_index < len(bits):
                    bit = int(bits[data_index])
                    img_copy[y, x, c] = (img_copy[y, x, c] & mask) | (bit << (bit_position - 1))
                    data_index += 1
                else:
                    return img_copy
    return img_copy

def extract_lbs(image, bit_position=1):
    bits = ''
    h, w, _ = image.shape

    if not (1 <= bit_position <= 8):
        raise ValueError("Vị trí bit phải nằm trong khoảng 1 đến 8.")

    for y in range(h):
        for x in range(w):
            for c in range(3):
                bit = (image[y, x, c] >> (bit_position - 1)) & 1
                bits += str(bit)
                if bits.endswith('1111111111111110'):
                    return bits_to_text(bits)
    return bits_to_text(bits)


def embed_sw_block(image, watermark_text, block_size=4):
    bits = text_to_bits(watermark_text)
    data_index = 0
    bw_img = cv2.threshold(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), 127, 1, cv2.THRESH_BINARY)[1]
    img_copy = bw_img.copy()
    h, w = img_copy.shape

    for i in range(0, h - block_size + 1, block_size):
        for j in range(0, w - block_size + 1, block_size):
            block = img_copy[i:i+block_size, j:j+block_size]
            if data_index >= len(bits):
                break
            bit_to_hide = int(bits[data_index])
            total_black = np.sum(block)  # 0=trắng, 1=đen nên tổng = số điểm đen
            parity = total_black % 2
            if parity != bit_to_hide:
                # Đảo một điểm bất kỳ trong block
                y, x = np.where(block == block[0, 0])
                block[y[0], x[0]] = 1 - block[y[0], x[0]]
            img_copy[i:i+block_size, j:j+block_size] = block
            data_index += 1

    # Trả về ảnh RGB cùng kích thước để so sánh với ảnh gốc
    binary_img_rgb = cv2.cvtColor((img_copy * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    return binary_img_rgb

def extract_sw_block(image, block_size=4):
    bw_img = cv2.threshold(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), 127, 1, cv2.THRESH_BINARY)[1]
    h, w = bw_img.shape
    bits = ''

    for i in range(0, h - block_size + 1, block_size):
        for j in range(0, w - block_size + 1, block_size):
            block = bw_img[i:i+block_size, j:j+block_size]
            total_black = np.sum(block)
            bit = str(total_black % 2)
            bits += bit
            if bits.endswith('1111111111111110'):
                return bits_to_text(bits)
    return bits_to_text(bits)

# ==== WU-LEE ====
def embed_wulee(image, watermark_text, delta=10):
    bits = text_to_bits(watermark_text)
    data_index = 0
    gray_img = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    img_copy = gray_img.copy()

    h, w = img_copy.shape
    for y in range(h):
        for x in range(w):
            if data_index >= len(bits):
                break
            bit = int(bits[data_index])
            if bit == 1:
                img_copy[y, x] = min(img_copy[y, x] + delta, 255)
            else:
                img_copy[y, x] = max(img_copy[y, x] - delta, 0)
            data_index += 1

    return cv2.cvtColor(img_copy, cv2.COLOR_GRAY2RGB)

def extract_wulee(image, original_image, delta=10):
    gray_embed = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray_orig = cv2.cvtColor(original_image, cv2.COLOR_RGB2GRAY)

    h, w = gray_embed.shape
    bits = ''
    for y in range(h):
        for x in range(w):
            diff = int(gray_embed[y, x]) - int(gray_orig[y, x])
            if abs(diff) >= delta:
                bits += '1' if diff > 0 else '0'
                if bits.endswith('1111111111111110'):
                    return bits_to_text(bits)
    return bits_to_text(bits)

import numpy as np

# === PCT===
def embed_pct(image, watermark_text, alpha=10):
    bits = text_to_bits(watermark_text)
    data_index = 0
    img_copy = image.copy()

    h, w, _ = img_copy.shape
    for y in range(h):
        for x in range(w):
            if data_index >= len(bits):
                break
            bit = int(bits[data_index])
            R, G, B = map(int, img_copy[y, x])

            if bit == 1:
                while R - B <= alpha and R < 255:
                    R += 1
                while R - B <= alpha and B > 0:
                    B -= 1
            else:
                while R - B >= -alpha and B < 255:
                    B += 1
                while R - B >= -alpha and R > 0:
                    R -= 1

            img_copy[y, x] = [R, G, B]
            data_index += 1
    return img_copy

def extract_pct(image, original_image, alpha=10):
    bits = ''
    h, w, _ = image.shape
    for y in range(h):
        for x in range(w):
            R_embed = int(image[y, x, 0])
            B_embed = int(image[y, x, 2])
            diff = R_embed - B_embed

            if diff > alpha:
                bits += '1'
            elif diff < -alpha:
                bits += '0'
            else:
                continue  # không chắc chắn, bỏ qua

            if bits.endswith('1111111111111110'):
                return bits_to_text(bits)
    return bits_to_text(bits)




# ==== DWT ====
def embed_dwt(image, watermark_text):
    bits = text_to_bits(watermark_text)
    data_index = 0
    img_copy = image.copy().astype(np.float32)
    h, w, _ = img_copy.shape  # Lưu kích thước gốc để resize lại sau
    for channel in range(3):  # R, G, B
        cA, (cH, cV, cD) = pywt.dwt2(img_copy[:, :, channel], 'haar')
        cD_flat = cD.flatten()
        for i in range(len(cD_flat)):
            if data_index < len(bits):
                cD_flat[i] = (int(cD_flat[i]) & 254) | int(bits[data_index])
                data_index += 1
        cD_mod = np.reshape(cD_flat, cD.shape)
        idwt = pywt.idwt2((cA, (cH, cV, cD_mod)), 'haar')
        img_copy[:, :, channel] = idwt[:h, :w]  # Cắt lại cho khớp kích thước
        if data_index >= len(bits):
            break
    return np.uint8(np.clip(img_copy, 0, 255))


def extract_dwt(image):
    bits = ''
    for channel in range(3):
        cA, (cH, cV, cD) = pywt.dwt2(image[:, :, channel].astype(np.float32), 'haar')
        cD_flat = cD.flatten()
        for val in cD_flat:
            bits += str(int(val) & 1)
            if bits.endswith('1111111111111110'):
                return bits_to_text(bits)
    return bits_to_text(bits)

# ==== DCT ====
import scipy.fftpack

def embed_dct(image, watermark_text):
    bits = text_to_bits(watermark_text)
    data_index = 0
    img_copy = image.copy().astype(np.float32)

    h, w, _ = img_copy.shape
    for ch in range(3):  # R, G, B
        for i in range(0, h, 8):
            for j in range(0, w, 8):
                if i + 8 > h or j + 8 > w:
                    continue
                block = img_copy[i:i+8, j:j+8, ch]
                dct_block = scipy.fftpack.dct(scipy.fftpack.dct(block.T, norm='ortho').T, norm='ortho')
                if data_index < len(bits):
                    coeff = dct_block[4, 4]
                    dct_block[4, 4] = (int(coeff) & 254) | int(bits[data_index])
                    data_index += 1
                idct_block = scipy.fftpack.idct(scipy.fftpack.idct(dct_block.T, norm='ortho').T, norm='ortho')
                img_copy[i:i+8, j:j+8, ch] = idct_block
                if data_index >= len(bits): break
            if data_index >= len(bits): break
        if data_index >= len(bits): break
    return np.uint8(np.clip(img_copy, 0, 255))

def extract_dct(image):
    bits = ''
    h, w, _ = image.shape
    for ch in range(3):
        for i in range(0, h, 8):
            for j in range(0, w, 8):
                if i + 8 > h or j + 8 > w:
                    continue
                block = image[i:i+8, j:j+8, ch].astype(np.float32)
                dct_block = scipy.fftpack.dct(scipy.fftpack.dct(block.T, norm='ortho').T, norm='ortho')
                bits += str(int(dct_block[4, 4]) & 1)
                if bits.endswith('1111111111111110'):
                    return bits_to_text(bits)
    return bits_to_text(bits)

def convert_rgb_to_binary_rgb(image):
    # Chuyển ảnh RGB sang ảnh nhị phân rồi về RGB để giữ shape (H, W, 3)
    bw_img = cv2.threshold(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), 127, 1, cv2.THRESH_BINARY)[1]
    binary_rgb = cv2.cvtColor((bw_img * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    return binary_rgb

# ==== PSNR & SSIM ====
def calculate_psnr(img1, img2):
    mse = np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')
    PIXEL_MAX = 255.0
    return round(20 * math.log10(PIXEL_MAX / math.sqrt(mse)), 2)

def calculate_ssim(img1, img2):
    if img1.shape != img2.shape:
        return None
    ssim_total = 0
    for i in range(3):
        ssim_total += ssim(img1[..., i], img2[..., i], data_range=255)
    return round(ssim_total / 3, 4)

# ==== Widgets & Giao diện ====
upload_btn = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
method_dropdown = widgets.Dropdown(options=["LSB", "LBS", "SW", "WU-LEE", "PCT", "DWT", "DCT"], value="LSB", description="Thuật toán:")
watermark_input = widgets.Text(value='', description='Watermark:')
embed_btn = widgets.Button(description='Nhúng watermark', button_style='success')
extract_btn = widgets.Button(description='Giải watermark', button_style='info')
output_box = widgets.Output()

# Thêm lựa chọn vị trí bit cho LBS
bit_position_dropdown = widgets.Dropdown(
    options=[1, 2, 3, 4, 5, 6, 7, 8],
    value=1,
    description="Vị trí bit(LBS):",
    layout=widgets.Layout(width='200px')
)


uploaded_img_rgb = None
watermarked_img = None

# ==== Callback ====
def on_upload_change(change):
    global uploaded_img_rgb
    output_box.clear_output()
    if not upload_btn.value:
        with output_box:
            print("Bạn chưa chọn ảnh.")
        return
    try:
        file_info = upload_btn.value[0]
        name = file_info['name']
        content = file_info['content']
        image_pil = Image.open(io.BytesIO(content)).convert("RGB")
        uploaded_img_rgb = np.array(image_pil)
        with output_box:
            clear_output()
            print(f"Ảnh đã chọn: {name}")
            plt.imshow(uploaded_img_rgb)
            plt.axis('off')
            plt.title("Ảnh gốc")
            plt.show()
    except Exception as e:
        with output_box:
            print("Không đọc được ảnh:", str(e))
            
# Hàm nhúng watermark
def on_embed_click(b):
    global watermarked_img
    output_box.clear_output()
    if uploaded_img_rgb is None:
        with output_box:
            print("Chưa có ảnh để nhúng.")
        return
    try:
        method = method_dropdown.value
        wm_text = watermark_input.value
        bit_pos = bit_position_dropdown.value
        
        if method_dropdown.value == "LSB":
            watermarked_img = embed_lsb(uploaded_img_rgb, watermark_input.value)
        elif method_dropdown.value == "LBS":
            watermarked_img = embed_lbs(uploaded_img_rgb, wm_text, bit_position=bit_pos)
        elif method_dropdown.value == "SW":
            watermarked_img = embed_sw_block(uploaded_img_rgb, watermark_input.value)
        elif method_dropdown.value == "WU-LEE":
            watermarked_img = embed_wulee(uploaded_img_rgb, wm_text)
        elif method_dropdown.value == "PCT":
            watermarked_img = embed_pct(uploaded_img_rgb, wm_text)
        elif method_dropdown.value == "DWT":
            watermarked_img = embed_dwt(uploaded_img_rgb, watermark_input.value)
        elif method_dropdown.value == "DCT":
            watermarked_img = embed_dct(uploaded_img_rgb, watermark_input.value)

        else:
            raise ValueError("Thuật toán chưa hỗ trợ.")

        if method == "SW":
            # So sánh với ảnh nhị phân gốc, không phải RGB
            binary_original = convert_rgb_to_binary_rgb(uploaded_img_rgb)
            psnr_score = calculate_psnr(binary_original, watermarked_img)
            ssim_score = calculate_ssim(binary_original, watermarked_img)
        else:
            psnr_score = calculate_psnr(uploaded_img_rgb, watermarked_img)
            ssim_score = calculate_ssim(uploaded_img_rgb, watermarked_img)

        with output_box:
            print(f"Nhúng watermark bằng {method_dropdown.value} thành công!")
            print(f"PSNR: {psnr_score} dB")
            print(f"SSIM: {ssim_score}")
            # Tạo ảnh hiển thị watermark chữ (không ảnh hưởng ảnh nhúng thực tế)
            display_img = watermarked_img.copy()
            cv2.putText(display_img, wm_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 0, 255), 2, cv2.LINE_AA)
            display_img_rgb = cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB)
            # Hiển thị ảnh gốc và ảnh nhúng có overlay chữ
            fig, axs = plt.subplots(1, 2, figsize=(10, 4))
            axs[0].imshow(uploaded_img_rgb)
            axs[0].set_title("Ảnh gốc")
            axs[0].axis('off')
            axs[1].imshow(watermarked_img)
            axs[1].set_title("Ảnh nhúng")
            axs[1].axis('off')
            plt.tight_layout()
            plt.show()
    except Exception as e:
        with output_box:
            print("Lỗi nhúng:", str(e))

# hàm giải watermark
def on_extract_click(b):
    if watermarked_img is None:
        with output_box:
            print("Chưa có ảnh đã nhúng.")
        return
    try:
        bit_pos = bit_position_dropdown.value
        if method_dropdown.value == "LSB":
            message = extract_lsb(watermarked_img)
        elif method_dropdown.value == "LBS":
            message = extract_lbs(watermarked_img, bit_position=bit_pos)
        elif method_dropdown.value == "SW":
            message = extract_sw_block(watermarked_img)
        elif method_dropdown.value == "WU-LEE":
            message = extract_wulee(watermarked_img, uploaded_img_rgb)
        elif method_dropdown.value == "PCT":
            message = extract_pct(watermarked_img, uploaded_img_rgb)
        elif method_dropdown.value == "DWT":
            message = extract_dwt(watermarked_img)
        elif method_dropdown.value == "DCT":
            message = extract_dct(watermarked_img)

        else:
            message = "Chưa hỗ trợ giải thuật toán này."

        with output_box:
            print(f"🔍 Watermark trích xuất ({method_dropdown.value}):", message)
    except Exception as e:
        with output_box:
            print("Lỗi giải mã:", str(e))

# ==== Hiển thị giao diện ====
upload_btn.observe(on_upload_change, names='value')
embed_btn.on_click(on_embed_click)
extract_btn.on_click(on_extract_click)

ui = widgets.VBox([
    widgets.HBox([widgets.Label("🖼Chọn ảnh:"), upload_btn]),
    method_dropdown,
    watermark_input,
    bit_position_dropdown,
    widgets.HBox([embed_btn, extract_btn]),
    output_box
])
display(ui)
